# Task 1: Logistic Regression for Binary Classification

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,roc_curve,roc_auc_score,confusion_matrix

# Load the dataset
I use the churn-bigml-20.csv as a dataset.

In [ ]:
df=pd.read_csv("C:/Users/Hp\Documents/My project/Codveda Projects/Dataset/Churn Prdiction Data/churn-bigml-20.csv")
df.head()
df.info()


# Handling the dataset
I checked the missing values and duplicates.

In [ ]:
# handling dataset
print("Missing values before handling:", df.isnull().sum())
print("Duplicate before handling:",df.duplicated().sum())

# Encoding Categorical Features

In [ ]:
#encoding
if "State" in df.columns:
  df=pd.get_dummies(df,columns=["State"],drop_first=False)

df["International plan"] = df["International plan"].replace({"No":0,"Yes":1})
df["Voice mail plan"] = df["Voice mail plan"].replace({"No":0,"Yes":1})
df["Churn"] = df["Churn"].replace({False:0,True:1})
df.head()


# Feature and Target separation

In [ ]:
# feature and target separation
X=df.drop(columns=["Churn"],axis=1,errors="ignore")
y=df["Churn"]

# Train and Test split

In [ ]:
# Train-Test split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,shuffle=True,random_state=42)

# Feature Scaling

In [ ]:
# Feature Scaling
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

# Train with Logistic Regression

In [ ]:
# Train logistic regression
model=LogisticRegression(max_iter=1000,class_weight="balanced")
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
print(y_pred[:10])
print(y_test[:10])


# Interpretation of Model Coefficients and Odds Ratios

Logistic Regression coefficients were interpreted using **odds ratios** obtained by exponentiating the model coefficients:

Odds Ratio = exp(coefficient)

- An odds ratio **greater than 1** indicates the feature increases the likelihood of churn.
- An odds ratio **less than 1** indicates the feature reduces the likelihood of churn.
- An odds ratio **equal to 1** indicates little or no effect.

Based on the model results, **Customer service calls** showed a strong positive relationship with churn, while features such as **Voice mail plan** reduced churn likelihood. Usage-related variables also contributed to churn prediction.


In [ ]:
# interprate model cofficent and odd ratio
model_cofficients=model.coef_[0]
model_intercept=model.intercept_
model_odds_ratio=np.exp(model_cofficients)
for feature,coff,odds in zip(X.columns,model_cofficients,model_odds_ratio):
  print(feature,coff,odds)


In [ ]:
# Evaluation
accuracy=accuracy_score(y_test,y_pred)
precision=precision_score(y_test,y_pred)
recall=recall_score(y_test,y_pred)
cm=confusion_matrix(y_test,y_pred)
print("Accuracy:",accuracy)
print("Precision:",precision)
print("Recall:",recall)
print("Confusion matrix:",cm)

# Model Evaluation Summary

The Logistic Regression model achieved:

- **Accuracy:** 72.4%  
- **Precision:** 25%  
- **Recall:** 73.3%  
- **AUC:** 0.783  

## Confusion Matrix
[[86 33]  
 [ 4 11]]

- Correctly identified **11 churners** and missed only **4**, giving strong recall.
- Precision is low because the model produced **33 false positives**, meaning some non-churn customers were predicted as churners.

## Interpretation
- The model performs well at **detecting churners** (high recall), which is important in churn prediction.
- Lower precision indicates a trade-off of more false alarms.
- **AUC of 0.783** shows good ability to distinguish churning from non-churning customers.

## Conclusion
The model shows promising predictive performance, especially for identifying potential churners, though precision can be improved through threshold tuning or class balancing.

# ROC Curve Visualization

The ROC curve was used to evaluate the model’s classification performance across different decision thresholds.

- **AUC Score:** 0.783

## Interpretation
- The ROC curve lies above the diagonal baseline, showing the model performs better than random guessing.
- An AUC of **0.783** indicates good ability to distinguish churning customers from non-churning customers.
- The curve shows a reasonable trade-off between **True Positive Rate (Recall)** and **False Positive Rate**.

## Conclusion
The ROC-AUC result suggests the Logistic Regression model has good discriminatory power and is effective for churn prediction.

In [ ]:
# visualization
probs=model.predict_proba(X_test)[:,1]
fpr,tpr,thresholds=roc_curve(y_test,probs)
auc=roc_auc_score(y_test,probs)
print("AUC",auc)

plt.plot(fpr,tpr,label=f"AUC = {auc:.3f}")
plt.plot([0,1],[0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

